In [1]:
!pip3 install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


# Recent bench reports — historical comparison

A focused view of how the per-algorithm wall-clock has evolved across the
last few `./build.sh bench` runs. The notebook is intentionally narrow:
it answers _"did the last commit regress or improve performance?"_ for
the length buckets that matter most (LONG, XL, XXL) and the four
algorithms we care about most.

Sister notebook: `bioseq/RESEARCH/BENCHMARKING_SPEEDUP/view_report.ipynb`
renders one full report in detail. This one trades depth for breadth so
regressions across the last few runs are obvious at a glance.

## What this notebook does

1. Walks `main/reporting/` and picks the **most recent up to 5** reports,
   ranked by the `<N>` suffix of `report_build_<N>.json` (higher `N` =
   newer; the bench writer auto-increments this counter).
2. Loads each report's `seq_length_vs_speed_mean` chart and computes the
   mean wall-clock per pair for every (algorithm, length-bucket) cell.
3. Renders one table per length bucket (LONG, XL, XXL) where rows are
   the four headline algorithms and columns are the selected reports in
   chronological order (oldest on the left, newest on the right).
4. The column header has two levels: the upper level groups consecutive
   reports that share a commit hash (so a re-run on the same commit
   reads as one block), and the lower level shows the report number.
   Full commit hashes are listed at the bottom — the header itself
   stays compact.
5. Cells are tinted **green** when the value is more than `tolerance_pct`
   _faster_ than the cell to its left, **red** when it is more than
   `tolerance_pct` _slower_. The first column has no neighbour to
   compare to and is left uncoloured. `tolerance_pct` is loaded from
   `regression_thresholds.json` so this view stays in sync with the
   regression check that the bench harness already enforces.

## Why this view exists

After landing a perf-sensitive change you want a one-screen answer to
_"did anything regress?"_ before opening the much larger per-report
notebook. Spotting a sudden red column in the LONG/XL/XXL tables is a
lot faster than staring at 30+ Plotly charts.

In [2]:
# ----- configuration + threshold load -----
# REPORTING_DIR is where ./build.sh bench writes report_build_<N>.json.
# THRESHOLDS_PATH is the same JSON the regression checker reads, so
# tolerance_pct used to colour cells matches what the bench enforces.

import json
import pathlib
import re

import pandas as pd
from IPython.display import display, Markdown

REPORTING_DIR   = pathlib.Path('../reporting/bioseq_bench')
THRESHOLDS_PATH = pathlib.Path('regression_thresholds.json')
MAX_REPORTS     = 10

with open(THRESHOLDS_PATH, 'r') as fh:
    thresholds_doc = json.load(fh)
TOLERANCE_PCT = float(thresholds_doc.get('tolerance_pct', 2.0))
print(f'tolerance_pct (from regression_thresholds.json): {TOLERANCE_PCT}%')

tolerance_pct (from regression_thresholds.json): 2.0%


In [3]:
# Discover report_build_<N>.json files. Pick the 5 with the highest
# <N> (newest by index), then re-sort ascending so the leftmost column
# in every table is the *oldest* of the selected reports — reading
# left-to-right then matches the order the runs happened.

report_pattern = re.compile(r'^report_build_(\d+)\.json$')
candidates = []
for path in REPORTING_DIR.glob('report_build_*.json'):
    m = report_pattern.match(path.name)
    if m:
        candidates.append((int(m.group(1)), path))
if not candidates:
    raise SystemExit(f'No report_build_*.json files found in {REPORTING_DIR.resolve()}')
candidates.sort(key=lambda t: t[0], reverse=True)
candidates = candidates[:MAX_REPORTS]
candidates.sort(key=lambda t: t[0])

reports = []
for idx, path in candidates:
    with open(path, 'r') as fh:
        data = json.load(fh)
    reports.append({
        'index':  idx,
        'path':   path,
        'commit': data.get('commit_hash', 'unknown'),
        'data':   data,
    })

# Assign one stable label per unique commit hash, ordered by first
# appearance in the chronological list. Reports that share a hash
# share a label; pandas MultiIndex will visually merge adjacent
# identical labels in the column header.
commit_order = []
commit_to_label = {}
for r in reports:
    c = r['commit']
    if c not in commit_to_label:
        commit_order.append(c)
        commit_to_label[c] = f'commit {len(commit_order)}'
    r['commit_label'] = commit_to_label[c]

print(f'Loaded {len(reports)} report(s):')
for r in reports:
    short = (r['commit'][:12] + '\u2026') if len(r['commit']) > 12 else r['commit']
    print(f'  report {r["index"]:>3}  {r["commit_label"]:<10}  {short:<14}  ({r["path"].name})')

Loaded 10 report(s):
  report  12  commit 1    8911f5f107e0…   (report_build_12.json)
  report  13  commit 1    8911f5f107e0…   (report_build_13.json)
  report  14  commit 1    8911f5f107e0…   (report_build_14.json)
  report  15  commit 2    35b9e5bea6d4…   (report_build_15.json)
  report  16  commit 2    35b9e5bea6d4…   (report_build_16.json)
  report  17  commit 2    35b9e5bea6d4…   (report_build_17.json)
  report  18  commit 3    bfda2ebce1a2…   (report_build_18.json)
  report  19  commit 3    bfda2ebce1a2…   (report_build_19.json)
  report  20  commit 3    bfda2ebce1a2…   (report_build_20.json)
  report  21  commit 4    1ff1f692b415…   (report_build_21.json)


In [4]:
# ----- algorithm + bucket definitions + helpers -----
# (canonical name in report.json, display label in the table row).
# The canonical names match the labels matrix_runner.cppm uses when it
# emits seq_length_vs_speed_mean traces.

ALGORITHMS = [
    ('minimap2 (DP chaining)',
     'minimap2 (DP, non-RMQ)'),
    ('bioseq KSW2 + scalar-DP + scalar-mm',
     'KSW2 + bioseq scalar + scalar minimizers'),
    ('bioseq KSW2 + SIMD-DP + two-stack-nthash-mm',
     'KSW2 + bioseq SIMD-DP + two-stack-nthash'),
    ('bioseq Block + SIMD-DP + two-stack-nthash-mm',
     'Block aligner + bioseq SIMD-DP + two-stack-nthash'),
]

# Length buckets we care about for the historical view. The boundaries
# match the bench harness's bucket definitions in matrix_runner.cppm.
LONG_BUCKET = (10_000,     1_000_000)
XL_BUCKET   = (5_000_000,  15_000_000)
XXL_BUCKET  = (50_000_000, 150_000_000)


def bucket_mean(report_data, algorithm_name, lo, hi):
    '''Mean wall-clock (ms) across all (query, target) pairs whose
    query length falls in [lo, hi). Returns None if the algorithm
    has no point in the bucket — typical when XL/XXL was not part
    of that run, or the algorithm is not present in the report.'''
    chart = report_data.get('charts', {}).get('seq_length_vs_speed_mean')
    if not chart:
        return None
    for trace in chart.get('plotly', {}).get('data', []):
        if trace.get('name') != algorithm_name:
            continue
        xs = trace.get('x', [])
        ys = trace.get('y', [])
        in_bucket = [y for x, y in zip(xs, ys)
                     if y is not None and lo <= x < hi]
        if not in_bucket:
            return None
        return sum(in_bucket) / len(in_bucket)
    return None


def build_bucket_dataframe(bucket_lo, bucket_hi):
    '''DataFrame: rows = ALGORITHMS, columns = MultiIndex(commit, report).'''
    columns = pd.MultiIndex.from_tuples(
        [(r['commit_label'], f'report {r["index"]}') for r in reports],
        names=['Commit', 'Report'],
    )
    rows = []
    index = []
    for canonical, display_name in ALGORITHMS:
        index.append(display_name)
        rows.append([bucket_mean(r['data'], canonical, bucket_lo, bucket_hi)
                     for r in reports])
    return pd.DataFrame(rows, index=index, columns=columns)


def colour_history(df, tolerance_pct):
    '''Tint each cell relative to the previous (left) cell in the same row.

    delta_pct = (current - prev) / prev * 100. Negative is faster
    (green), positive is slower (red). Cells inside the tolerance
    window stay uncoloured. The first column, and any cell whose
    left-hand neighbour is missing, are left uncoloured.

    Colour intensity scales with how far past tolerance the delta is,
    capped at 20 percentage points beyond tolerance for readability.
    '''
    def row_styles(values):
        styles = ['' for _ in values]
        prev = None
        for i, value in enumerate(values):
            if pd.isna(value):
                continue
            if prev is not None and prev > 0:
                delta_pct = (value - prev) / prev * 100.0
                if delta_pct < -tolerance_pct:
                    intensity = min(1.0, (abs(delta_pct) - tolerance_pct) / 20.0)
                    rb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb({rb}, 255, {rb}); '
                                 f'color: black; font-weight: 600;')
                elif delta_pct > tolerance_pct:
                    intensity = min(1.0, (delta_pct - tolerance_pct) / 20.0)
                    gb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb(255, {gb}, {gb}); '
                                 f'color: black; font-weight: 600;')
            prev = value
        return styles
    return (df.style
              .apply(row_styles, axis=1)
              .format('{:.2f} ms', na_rep='\u2014')
              .set_table_styles([
                  {'selector': 'th',
                   'props': 'text-align: center; padding: 4px 10px;'},
                  {'selector': 'td',
                   'props': 'text-align: right; padding: 4px 10px; '
                            'font-variant-numeric: tabular-nums;'},
              ]))

## LONG bucket (10 kbp – 1 Mbp)

The most balanced regression signal: long enough that base-level
alignment dominates the wall-clock, short enough that every report
still has a healthy number of samples per (algorithm, error) cell, so
the means here are reasonably stable and a colour change usually
reflects a real speedup or regression rather than noise.

In [5]:
display(colour_history(build_bucket_dataframe(*LONG_BUCKET), TOLERANCE_PCT))

## XL bucket (5 Mbp – 15 Mbp)

Two samples per pair by default (one with `--fast`). Each pair takes
seconds of wall-clock, so a regression in this bucket is the single
biggest cost contributor to a typical full bench run. Colour changes
here matter even when the millisecond delta looks small in absolute
terms — the percentage swing is what the regression checker scores.

In [6]:
display(colour_history(build_bucket_dataframe(*XL_BUCKET), TOLERANCE_PCT))

## XXL bucket (50 Mbp – 150 Mbp)

Single-sample bucket — each cell is one measurement, so the noise
floor is high. Treat colouring here as a hint rather than a verdict:
a cell that flips between red and green across consecutive reports
is normal noise, while a genuine regression shows up as a sustained
red trend across two or more reports in a row.

In [7]:
display(colour_history(build_bucket_dataframe(*XXL_BUCKET), TOLERANCE_PCT))

## Commit hashes

Full commit hash for each `commit N` label used in the column headers
above. Use these to cross-reference `git show <hash>` (or open the
commit on GitHub) when a coloured cell points at a specific change.

In [8]:
lines = ['**Commit hashes:**', '']
for commit in commit_order:
    label = commit_to_label[commit]
    lines.append(f'* **{label}**: `{commit}`')
display(Markdown('\n'.join(lines)))

**Commit hashes:**

* **commit 1**: `8911f5f107e001397382521802b5390951487780`
* **commit 2**: `35b9e5bea6d45d0c5cac7cbd647fde8adcc26dbc`
* **commit 3**: `bfda2ebce1a2aea72d52089b4929209a074eee50`
* **commit 4**: `1ff1f692b41525471adf2ae507c963e1359fb9dd`

# IO benchmark history

A parallel view for the `bioseq_io` reader benchmarks. The structure
mirrors the aligner section above, but **the colouring sense is
inverted**: throughput is measured in MiB/s where _higher is better_,
so a cell that grew vs. its left neighbour is **green** (improvement)
and one that shrunk is **red** (regression).

Reports come from `main/reporting/bioseq_io_bench/report_build_<N>.json`
and follow the `bioseq-io-bench-report-v1` schema. One table per
length bucket; rows are the readers we track, columns are the most
recent reports in chronological order.

In [9]:
# ----- IO bench: configuration + threshold load -----
# IO_REPORTING_DIR is where the IO bench writes report_build_<N>.json.
# IO_THRESHOLDS_PATH is the same JSON the IO regression checker reads,
# but its baseline is loose (150% — calibrated for SIMD-encode vs raw
# byte copy). For human-friendly visual comparison we override with
# IO_DISPLAY_TOLERANCE_PCT so adjacent-cell deltas feel meaningful.

IO_REPORTING_DIR       = pathlib.Path('../reporting/bioseq_io_bench')
IO_THRESHOLDS_PATH     = pathlib.Path('../bioseq/io/io_regression_thresholds.json')
IO_MAX_REPORTS         = 15
IO_DISPLAY_TOLERANCE_PCT = 5.0  # override; the 150% rule-engine threshold is too loose for visual diffs

with open(IO_THRESHOLDS_PATH, 'r') as fh:
    io_thresholds_doc = json.load(fh)
IO_RULE_TOLERANCE_PCT = float(io_thresholds_doc.get('tolerance_pct', 150.0))
print(f'IO rule-engine tolerance_pct: {IO_RULE_TOLERANCE_PCT}% (loose by design)')
print(f'IO display tolerance_pct (used for cell colouring below): {IO_DISPLAY_TOLERANCE_PCT}%')

IO rule-engine tolerance_pct: 150.0% (loose by design)
IO display tolerance_pct (used for cell colouring below): 5.0%


In [10]:
# Discover IO report_build_<N>.json files. Same selection logic as the
# aligner section: take the most recent IO_MAX_REPORTS by index, then
# re-sort ascending so the leftmost column is the oldest.

io_candidates = []
for path in IO_REPORTING_DIR.glob('report_build_*.json'):
    m = report_pattern.match(path.name)
    if m:
        io_candidates.append((int(m.group(1)), path))
if not io_candidates:
    raise SystemExit(f'No report_build_*.json files found in {IO_REPORTING_DIR.resolve()}')
io_candidates.sort(key=lambda t: t[0], reverse=True)
io_candidates = io_candidates[:IO_MAX_REPORTS]
io_candidates.sort(key=lambda t: t[0])

io_reports = []
for idx, path in io_candidates:
    try:
        with open(path, 'r') as fh:
            data = json.load(fh)
    except (json.JSONDecodeError, OSError) as exc:
        # Skip corrupt/incomplete reports — early bench iterations
        # sometimes wrote partial files. The viewer should keep
        # going rather than crash.
        print(f'  skip report {idx} ({path.name}): {exc}')
        continue
    io_reports.append({
        'index':  idx,
        'path':   path,
        'commit': data.get('commit_hash', 'unknown'),
        'data':   data,
    })

# Independent commit-label scheme for the IO section. The aligner and
# IO benches don't have to share commits, so we number them separately
# (avoids 'commit 7 in this section, commit 2 in that one' confusion).
io_commit_order = []
io_commit_to_label = {}
for r in io_reports:
    c = r['commit']
    if c not in io_commit_to_label:
        io_commit_order.append(c)
        io_commit_to_label[c] = f'io-commit {len(io_commit_order)}'
    r['commit_label'] = io_commit_to_label[c]

print(f'Loaded {len(io_reports)} IO report(s):')
for r in io_reports:
    short = (r['commit'][:12] + '\u2026') if len(r['commit']) > 12 else r['commit']
    print(f'  report {r["index"]:>3}  {r["commit_label"]:<14}  {short:<14}  ({r["path"].name})')

Loaded 15 IO report(s):
  report  18  io-commit 1     3daa85a15ea5…   (report_build_18.json)
  report  19  io-commit 1     3daa85a15ea5…   (report_build_19.json)
  report  20  io-commit 1     3daa85a15ea5…   (report_build_20.json)
  report  21  io-commit 1     3daa85a15ea5…   (report_build_21.json)
  report  22  io-commit 1     3daa85a15ea5…   (report_build_22.json)
  report  23  io-commit 1     3daa85a15ea5…   (report_build_23.json)
  report  24  io-commit 1     3daa85a15ea5…   (report_build_24.json)
  report  25  io-commit 1     3daa85a15ea5…   (report_build_25.json)
  report  26  io-commit 2     655f2d5f33bb…   (report_build_26.json)
  report  27  io-commit 2     655f2d5f33bb…   (report_build_27.json)
  report  28  io-commit 2     655f2d5f33bb…   (report_build_28.json)
  report  29  io-commit 2     655f2d5f33bb…   (report_build_29.json)
  report  30  io-commit 2     655f2d5f33bb…   (report_build_30.json)
  report  31  io-commit 2     655f2d5f33bb…   (report_build_31.json)
  report  

In [11]:
# ----- IO reader + length-bucket definitions + helpers -----
# Trace names match exactly the strings the IO bench writes into
# charts.throughput_by_length.plotly.data[*].name (see
# main/bioseq/io/bench/chart_builder.cppm or the equivalent emitter).

IO_READERS = [
    ('bioseq_io_sync_fasta (fasta)',   'bioseq_io_sync (fasta)'),
    ('bioseq_io_mmap_fasta (fasta)',   'bioseq_io_mmap (fasta)'),
    ('bioseq_io_async4_fasta (fasta)', 'bioseq_io_async4 (fasta)'),
    ('bioseq_io_sync_fastq (fastq)',   'bioseq_io_sync (fastq)'),
    ('bioseq_io_mmap_fastq (fastq)',   'bioseq_io_mmap (fastq)'),
    ('kseqpp (fasta)',                 'kseqpp (fasta)'),
    ('kseqpp (fastq)',                 'kseqpp (fastq)'),
    ('naive_ifstream (fasta)',         'naive_ifstream (fasta)'),
    ('naive_ifstream (fastq)',         'naive_ifstream (fastq)'),
]

# X-axis labels of the throughput_by_length plotly bar chart, in the
# order the IO emitter writes them. We render one table per bucket.
IO_LENGTH_BUCKETS = [
    'short (~200bp)',
    'medium (~2kbp)',
    'long (~50kbp)',
    'very long (~1Mbp)',
    'real human genome (~234MiB)',
]


def io_throughput(report_data, reader_name, bucket_label):
    '''MiB/s for a (reader, length-bucket) cell. Returns None when
    the reader is missing from the report or has no point at that
    bucket. A 0.0 value is also treated as missing — the IO bench
    writes 0.0 for cells that were skipped (e.g. fastq has no
    234MiB human-genome corpus).'''
    chart = report_data.get('charts', {}).get('throughput_by_length')
    if not chart:
        return None
    for trace in chart.get('plotly', {}).get('data', []):
        if trace.get('name') != reader_name:
            continue
        xs = trace.get('x', [])
        ys = trace.get('y', [])
        for x, y in zip(xs, ys):
            if x == bucket_label:
                if y is None or y == 0.0:
                    return None
                return float(y)
        return None
    return None


def build_io_dataframe(bucket_label):
    '''DataFrame: rows = IO_READERS, columns = MultiIndex(commit, report).'''
    columns = pd.MultiIndex.from_tuples(
        [(r['commit_label'], f'report {r["index"]}') for r in io_reports],
        names=['Commit', 'Report'],
    )
    rows = []
    index = []
    for canonical, display_name in IO_READERS:
        index.append(display_name)
        rows.append([io_throughput(r['data'], canonical, bucket_label)
                     for r in io_reports])
    return pd.DataFrame(rows, index=index, columns=columns)


def colour_io_history(df, tolerance_pct):
    '''Same neighbour-comparison logic as colour_history, but with
    INVERTED sign: throughput is higher-is-better, so growth (delta
    > tolerance) is green and shrinkage (delta < -tolerance) is red.'''
    def row_styles(values):
        styles = ['' for _ in values]
        prev = None
        for i, value in enumerate(values):
            if pd.isna(value):
                continue
            if prev is not None and prev > 0:
                delta_pct = (value - prev) / prev * 100.0
                if delta_pct > tolerance_pct:
                    intensity = min(1.0, (delta_pct - tolerance_pct) / 20.0)
                    rb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb({rb}, 255, {rb}); '
                                 f'color: black; font-weight: 600;')
                elif delta_pct < -tolerance_pct:
                    intensity = min(1.0, (abs(delta_pct) - tolerance_pct) / 20.0)
                    gb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb(255, {gb}, {gb}); '
                                 f'color: black; font-weight: 600;')
            prev = value
        return styles
    return (df.style
              .apply(row_styles, axis=1)
              .format('{:.1f} MiB/s', na_rep='\u2014')
              .set_table_styles([
                  {'selector': 'th',
                   'props': 'text-align: center; padding: 4px 10px;'},
                  {'selector': 'td',
                   'props': 'text-align: right; padding: 4px 10px; '
                            'font-variant-numeric: tabular-nums;'},
              ]))

## IO short bucket (~200 bp records)

Per-record dispatch overhead dominates here — 40 000 records per
iteration, so any change in the parser hot path or the per-record
allocation pattern shows up first.

In [12]:
display(colour_io_history(build_io_dataframe('short (~200bp)'), IO_DISPLAY_TOLERANCE_PCT))

## IO medium bucket (~2 kbp records)

Balanced view: per-record overhead has dropped enough that the
encoder/copy throughput starts to matter, but record count is still
high enough (5 000) that the mean is stable.

In [13]:
display(colour_io_history(build_io_dataframe('medium (~2kbp)'), IO_DISPLAY_TOLERANCE_PCT))

## IO long bucket (~50 kbp records)

Encoder-bound regime: per-record dispatch is negligible, the body
copy/encode throughput is what the numbers track. This is the
bucket most sensitive to SIMD-kernel changes.

In [14]:
display(colour_io_history(build_io_dataframe('long (~50kbp)'), IO_DISPLAY_TOLERANCE_PCT))

## IO very long bucket (~1 Mbp records)

Almost pure streaming throughput — 15 records per iteration, each
around a megabase. Cache effects on the encoder's working set start
to be visible, but per-record overhead is fully amortised.

In [15]:
display(colour_io_history(build_io_dataframe('very long (~1Mbp)'), IO_DISPLAY_TOLERANCE_PCT))

## IO real human genome bucket (~234 MiB FASTA)

Single-corpus, single-record-equivalent end-to-end run on a real
human-genome FASTA. Closest proxy for actual production workloads.
Only the FASTA readers populate this row — the FASTQ readers have
no equivalent corpus, so their cells will appear empty.

In [16]:
display(colour_io_history(build_io_dataframe('real human genome (~234MiB)'), IO_DISPLAY_TOLERANCE_PCT))

In [17]:
display(colour_io_history(build_io_dataframe('real human fastq (~6GiB)'), IO_DISPLAY_TOLERANCE_PCT))

## IO commit hashes

Full commit hash for each `io-commit N` label used in the IO
section's column headers above.

In [18]:
io_lines = ['**IO commit hashes:**', '']
for commit in io_commit_order:
    label = io_commit_to_label[commit]
    io_lines.append(f'* **{label}**: `{commit}`')
display(Markdown('\n'.join(io_lines)))

**IO commit hashes:**

* **io-commit 1**: `3daa85a15ea51654ebb603e425a60abf0725a5e3`
* **io-commit 2**: `655f2d5f33bbde491bb473886564cba894684334`